# Saving & loading cases for Nemo

Write a solved network back to the UI's native YAML (case **+** result data), then read it in again.

In [ ]:
import sys, os, tempfile, yaml
_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "nefes")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np

from nefes.shell import Network
from nefes.elements import catalog as cat
from nefes.thermo.configure import perfect_gas
from nefes.io import load_case
from nefes.perturbation import forced_response, PerturbationBC

OUT = tempfile.mkdtemp()   # files land here; point the UI at them
OUT

## Save the case with chosen result fields

`Solution.to_yaml` embeds the picked fields as datasets; `node_data=True` also writes incident-edge means so nodes can be colored too.

In [ ]:
net = load_case("converging_nozzle.yaml")
sol = net.solve()

path = os.path.join(OUT, "nozzle_solved.yaml")
sol.to_yaml(path, fields=["mdot", "p", "M", "p_t"], node_data=True)

doc = yaml.safe_load(open(path).read())
for ds in doc["data"]["datasets"]:
    info = {e["label"]: e["value"] for e in ds.get("info", [])}
    print(ds["name"], info)
    print("  ", [f"{it['name']} ({it['target']})" for it in ds["items"]])

## Round-trip

Reload the saved file and re-solve — the network is recovered exactly (ids, ports, layout preserved).

In [ ]:
net2 = load_case(path)
sol2 = net2.solve()
print("identical:", np.allclose(sol.field("p"), sol2.field("p")))

## Acoustic frequency snapshots

A forced response writes one named dataset per frequency (magnitude + phase of each quantity).

In [ ]:
duct = Network(perfect_gas(287.0, 1.4), p_ref=101325.0, T_ref=300.0, mdot_ref=5.0)
duct.add(cat.total_pressure_inlet(104000.0, 300.0, perturbation_bc=PerturbationBC.anechoic(driven=("acoustic",))))
duct.add(cat.duct(0.5))
duct.add(cat.pressure_outlet(101325.0, 300.0, perturbation_bc=PerturbationBC.open_end()))
duct.connect(0, 1, 0.05); duct.connect(1, 2, 0.05)
dsol = duct.solve()

fr = forced_response(dsol.problem, dsol.x, np.array([120.0, 360.0, 600.0]))
apath = os.path.join(OUT, "duct_acoustic.yaml")
dsol.to_yaml(apath, fields=["p"], forced=fr, forced_freqs=[120.0, 600.0], forced_fields=("p", "u"))

print([d["name"] for d in yaml.safe_load(open(apath).read())["data"]["datasets"]])

## Build in Python → export a case

No provenance: ids and a left-to-right layout are synthesized. `Network.to_yaml` writes the case only (no results).

In [ ]:
g = Network(perfect_gas(287.0, 1.4), p_ref=101325.0, T_ref=300.0, mdot_ref=5.0)
g.add(cat.mass_flow_inlet(4.0, 300.0)); g.add(cat.duct(0.4)); g.add(cat.pressure_outlet(101325.0, 300.0))
g.connect(0, 1, 0.03); g.connect(1, 2, 0.03)

g.to_yaml(os.path.join(OUT, "built_in_python.yaml"), title="Python-built case")